In [ ]:
# Kaggle does not ship bitsandbytes, which the 4-bit config needs.
# Run this cell first, then restart the session if transformers was already imported.
!pip install -q "bitsandbytes>=0.46.1" accelerate

In [ ]:
import torch

# Transformers imports
# Every `transformers` import used by this notebook belongs in this cell.
# If a later step needs another Transformers component, add it here.
from transformers import (
    # Tokenizer and model: load the NLLB tokenizer and the seq2seq base model
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    # Data collation: pad English inputs and French labels per batch
    DataCollatorForSeq2Seq,
    # Training: configure hyperparameters and run the seq2seq training loop
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    # Model configuration and quantization: base config, NLLB tokenizer, and 4-bit settings
    AutoConfig,
    NllbTokenizer,
    BitsAndBytesConfig,
)

# Quantization configuration for Qlora

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.float32,
)

# Load the quantized NLLB model and tokenizer

In [ ]:
model_name = "facebook/nllb-200-distilled-1.3B"

config = AutoConfig.from_pretrained(model_name)

tokenizer = NllbTokenizer.from_pretrained(
    model_name,
    src_lang="eng_Latn",
    tgt_lang="fra_Latn",
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
)